# ДЗ-3. Свой A/B-тест

На семинарах С7 и С8 мы разбирали, как планировать и анализировать A/B-тесты. Теперь — самостоятельный анализ.

Представим, что вы аналитик ПРАЙМ. Команда продукта запустила акцию на 6 000 подписчиков: половина попала в тест, половина — в контроль. Эксперимент длился четыре недели, данные собраны. Продакт спрашивает: «Ну что, запускаем на всех?»

Ваша задача — разобраться, что получилось, и ответить ему. Получить `p < 0.05` и объявить эксперимент успешным недостаточно: нужно проверить, что он прошёл корректно, оценить эффект, посмотреть на побочные метрики и прикинуть, сколько ПРАЙМ на этом заработает или потеряет. Вполне может оказаться, что акцию лучше не запускать, — это тоже нормальный результат, если он следует из данных.

У каждого своя акция и свои данные: номер варианта берётся из `prime.hw3_variants`, как в хвосте семинара С7.

| Часть | Что там |
|---|---|
| **1. Подготовка** | готовый код: окружение, ваш `pilot.py`, ваш вариант |
| **2. Дизайн-документ** | как вы спланировали бы этот эксперимент |
| **3. Разминка** | задачи 1–4: дописать один аргумент |
| **4. Дописываем код** | задачи 5–8: дописать строку |
| **5. Самостоятельные задачи** | задачи 9–12: код целиком |
| **6. Отчёт** | что скажем продакту |
| **7. Итог** | ваши ответы и проверка их формата |
| **8. Исследование** | по желанию — для 9 и 10 |

Части 1–7 дают до 8 баллов, исследование — ещё до двух. Разбалловка и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-3-пилот).

JupyterLab запускается из корня `prime-monitor` командой `uv run --project pilot jupyter lab` — в окружении с HypEx, как на семинарах.

## Часть 1. Подготовка

Здесь только готовый код — запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Окружение | проверяет HypEx 1.0.7, подключается к базе, подключает ваш `src/prime/pilot.py` |
| 1.2 Ваш вариант | номер варианта; пока там `None`, он берётся из базы по вашему логину |
| 1.3 Выгрузка | загружает ваш эксперимент: 6 000 строк, одна строка — один клиент |
| 1.4 Формат ответов | служебная функция для итога |

### 1.1 Окружение

Та же ячейка, что на семинаре С8, плюс импорт пяти функций из вашего `src/prime/pilot.py`: `n_per_group` и `mde_for_n` с семинара С7, `diff_ci`, `srm_pvalue` и `cuped` с семинара С8. Дальше задачи считают ими.

Ошибка про HypEx — ноутбук запущен не в окружении `pilot/`. Ошибка про `pilot.py` — файла нет или в нём не хватает функции, см. текст задания.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Окружение pilot/ не устанавливает пакет prime: у него pandas 3, а HypEx 1.0.7
# работает только с pandas 2.2. Поэтому src/ подключаем к пути сами —
# ищем папку src/prime вверх от текущей.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "prime").is_dir()), None)
if root is None:
    raise RuntimeError("Не нашёл src/prime. Запустите JupyterLab из корня prime-monitor:\n"
                       "uv run --project pilot jupyter lab")
sys.path.insert(0, str(root / "src"))

import hypex

if hypex.__version__ != "1.0.7":
    raise RuntimeError(f"Нужен HypEx 1.0.7, а запущен {hypex.__version__} из {sys.executable}.\n"
                       "Закройте JupyterLab и запустите из корня prime-monitor: uv run --project pilot jupyter lab")

import numpy as np
import pandas as pd
from hypex import ABTest, Matching
from hypex.dataset import Dataset, FeatureRole, InfoRole, PreTargetRole, TargetRole, TreatmentRole
from hypex.utils.enums import ABNTestMethodsEnum
from scipy import stats
from sqlalchemy import text

from prime.config import get_engine

try:
    from prime.pilot import cuped, diff_ci, mde_for_n, n_per_group, srm_pvalue
except ImportError as e:
    raise ImportError(
        f"Не получилось импортировать функции из src/prime/pilot.py: {e}\n"
        "Нужны пять функций: n_per_group и mde_for_n (семинар С7, раздел 9), "
        "diff_ci, srm_pvalue и cuped (семинар С8, раздел 8)."
    ) from None

warnings.filterwarnings("ignore")          # HypEx шумит предупреждениями pandas
pd.set_option("display.width", 160)
engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def F(obj):
    """Результаты HypEx — объекты Dataset. Для pandas-операций берём .data."""
    return obj.data if hasattr(obj, "data") else obj


print("HypEx", hypex.__version__, "· pandas", pd.__version__)

### 1.2 Ваш вариант

Эту ячейку не меняйте: при проверке сюда подставляется номер другого варианта.

Исключение одно — сервер не отвечает (см. ячейку 1.3). Тогда впишите сюда номер своего варианта, например `VARIANT = 17`.

In [ ]:
VARIANT = None   # None — номер берётся из prime.hw3_variants по вашему логину

### 1.3 Выгрузка

Одна строка — один клиент.

| Колонки | Что это |
|---|---|
| `client_id`, `grp` | клиент и его группа: `test` или `control` |
| `age`, `tenure_days`, `kyc_level`, `members`, `acquisition_channel`, `time_zone` | возраст, дней с регистрации, уровень проверки личности, сколько людей в его подписке, канал привлечения, часовой пояс |
| `spend_prev2`, `purchases_prev2`, `sessions_prev2` | оборот в партнёрских сервисах, покупки и сессии в приложении за четыре недели **позапрошлого** периода |
| `spend_prev`, `purchases_prev`, `sessions_prev` | то же за четыре недели **прямо перед** экспериментом |
| `spend`, `purchases` | оборот и покупки за четыре недели эксперимента |
| `churned`, `tickets` | ушёл ли из подписки за время эксперимента (1 — да) и сколько раз писал в поддержку |

Основная метрика у всех вариантов — `spend`, оборот в партнёрских сервисах за четыре недели, ₽.

Если сервер не отвечает, поменяйте в первой строке `False` на `True` и впишите номер варианта в ячейку 1.2: те же данные скачаются из репозитория курса. Перед сдачей верните `False`.

In [ ]:
SERVER_IS_DOWN = False

RAW = ("https://raw.githubusercontent.com/tikhomirovd/python-for-ba-hse-2026/master/"
       "%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/%D0%BF%D0%B8%D0%BB%D0%BE%D1%82/")

if VARIANT is None:
    if SERVER_IS_DOWN:
        raise RuntimeError("Без сервера номер варианта не узнать: впишите его в ячейку 1.2.")
    VARIANT = int(q("select variant from prime.hw3_variants where login = current_user").iloc[0, 0])

if SERVER_IS_DOWN:
    offers = pd.read_csv(RAW + "hw3_offers.csv.gz")
    everyone = pd.read_csv(RAW + "hw3_pilot.csv.gz")
    mine = everyone[everyone["variant"] == VARIANT].copy()
else:
    offers = q("select * from prime.hw3_offers")
    mine = q("select * from prime.hw3_pilot where variant = :v", v=VARIANT)

# Один и тот же порядок строк при любом источнике: от него зависят бутстреп и мэтчинг.
mine["client_id"] = mine["client_id"].astype(str)
mine = mine.sort_values("client_id", ignore_index=True)
offer = offers.loc[offers["variant"] == VARIANT, "offer"].iloc[0]

test = mine[mine["grp"] == "test"]
control = mine[mine["grp"] == "control"]

print(f"вариант {VARIANT}: «{offer}»")
print(len(mine), "клиентов · test", len(test), "· control", len(control))

### 1.4 Формат ответов

Служебная ячейка: по ней итог в части 7 проверяет, что ответы записаны в нужном формате. Правильность она не проверяет. Просто запустите.

In [ ]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 3 знаков — p-значение",
    2: "float до 3 знаков — p-значение",
    3: "list [разность, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    4: "list [p t-критерия, p U-критерия] — float до 4 знаков",
    5: "list [отток: разница в п.п. до 2 знаков, p χ² до 3; обращения: разница до 3 знаков, p до 3]",
    6: "list названий метрик (str), может быть пустым",
    7: "list [θ до 3 знаков, снижение дисперсии в % до 1 знака, разность, нижняя, верхняя — до 2 знаков]",
    8: "float до 2 знаков — рубли",
    9: "list [нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    10: "list [название ковариаты (str), снижение дисперсии в % — float до 1 знака]",
    11: "list [ATE, нижняя граница, верхняя граница] — float до 2 знаков, рубли",
    12: "float до 2 знаков — рубли на клиента",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def dec(k):   # float, у которого не больше k знаков после запятой
        return lambda x: type(x) is float and round(x, k) == x

    def items(*checks):
        return lambda a: type(a) is list and len(a) == len(checks) and all(c(x) for c, x in zip(checks, a))

    def word(x) -> bool:
        return type(x) is str and x != ""

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: dec(3), 2: dec(3),
        3: items(dec(2), dec(2), dec(2)),
        4: items(dec(4), dec(4)),
        5: items(dec(2), dec(3), dec(3), dec(3)),
        6: lambda a: type(a) is list and all(word(x) for x in a),
        7: items(dec(3), dec(1), dec(2), dec(2), dec(2)),
        8: dec(2),
        9: items(dec(2), dec(2)),
        10: items(word, dec(1)),
        11: items(dec(2), dec(2), dec(2)),
        12: dec(2),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

## Часть 2. Дизайн-документ

Прежде чем смотреть на результаты, зафиксируем дизайн эксперимента — так, как если бы вы планировали его сами. Если писать дизайн после итогов, гипотеза незаметно подгоняется под результат. Если вы заполнили таблицу в хвосте семинара С7, перенесите её сюда.

Ячейка ниже считает, какой эффект эксперимент такого размера вообще способен заметить. Считает ваша `mde_for_n` по обороту за прошлые четыре недели: одно сравнение, α = 0,05, мощность 80 %.

In [ ]:
# Только предпериод: итогов пилота здесь нет.
n_each = len(mine) // 2
sigma = mine["spend_prev"].std()
rho = mine["spend_prev2"].corr(mine["spend_prev"])     # связь двух прошлых периодов

mde_raw = mde_for_n(sigma, n_each)
mde_cuped = mde_for_n(sigma * (1 - rho ** 2) ** 0.5, n_each)
three_pct = 0.03 * mine["spend_prev"].mean()

print(f"клиентов на группу: {n_each}, σ оборота: {sigma:.0f} ₽, ρ двух прошлых периодов: {rho:.3f}")
print(f"MDE без снижения дисперсии: {mde_raw:.0f} ₽")
print(f"MDE с CUPED:                {mde_cuped:.0f} ₽")
print(f"+3 % к среднему обороту:    {three_pct:.0f} ₽")

#### ✍️ Дизайн-документ

Заполните правую колонку для своей акции — как в разделе 8 семинара С7 и на слайде 32 лекции Л7. Многоточия замените своим текстом.

| Пункт | Наш эксперимент |
|---|---|
| гипотеза: что изменится, на сколько, за какое время | Новая главная страница приложения (партнёрские сервисы ближе и заметнее) поднимет средний оборот подписчика в партнёрских сервисах примерно на 190 ₽ (≈ 6 % от 3 198 ₽ в предпериоде) за 4 недели и при этом не увеличит отток и обращения в поддержку. Рост на 3 % (≈ 96 ₽) этот эксперимент надёжно не увидит — это фиксируем заранее. |
| единица рандомизации и почему именно она (подпиской делятся — колонка `members`) | Подписка целиком: владелец и все участники (`members` от 0 до 2) попадают в одну группу. Главную страницу видит каждый член семьи, а оборот и отток считаются на подписку; если делить по людям, в одной подписке окажутся обе версии, члены семьи будут видеть разное и обсуждать, эффекты «перетекут» между группами и оценка размоется. Метрики считаем на владельца подписки (`client_id`). |
| группы и доли | Две группы, 50/50: `test` — новая главная, `control` — старая; по 3 000 подписок (6 000 всего). |
| основная метрика | `spend` — оборот в партнёрских сервисах за 4 недели эксперимента, ₽ на клиента. |
| метрики-стражи: что не должно ухудшиться | Отток `churned` (доля ушедших из подписки) и обращения в поддержку `tickets` на клиента: новая главная может путать людей. Дополнительно смотрим `purchases` — число покупок. |
| α, мощность, MDE — числами из ячейки выше | α = 0,05, мощность 80 %, n = 3 000 на группу. По `mde_for_n`: MDE без снижения дисперсии — 232 ₽, с CUPED по прошлому обороту (ρ = 0,59 между двумя прошлыми периодами) — 188 ₽. +3 % к обороту — это 96 ₽, вдвое меньше MDE. |
| поправка на множественные сравнения | Холм по четырём метрикам: `spend`, `purchases`, `churned`, `tickets`. Метод выбран до итогов. |
| оценщик | Разность средних test − control с CUPED по `spend_prev` (оборот за 4 недели до старта), интервал — `diff_ci`; для сверки — обычная разность, бутстреп и HypEx. Ковариату выбираем сейчас, а не после итогов. |
| окно | 4 недели с момента включения новой главной; предпериод для CUPED — 4 недели прямо перед стартом. Промежуточных «подглядываний» с остановкой нет. |
| решающее правило: при каких результатах запускаем, дорабатываем, отказываемся | **Запускаем**, если одновременно: эффект с CUPED > 0 и значим после Холма (p < 0,05); отток и обращения значимо не выросли, а верхняя граница интервала по оттоку не выше +1 п.п.; в деньгах (задача 12) результат на клиента ≥ 0. **Дорабатываем и повторяем** на большей выборке, если эффект положительный, но не прошёл поправку, или стражи не исключают вреда больше порога, или деньги около нуля. **Отказываемся**, если какой-то страж значимо ухудшился или интервал эффекта лежит ниже MDE (эффект нулевой или отрицательный). |

## Часть 3. Разминка

Дальше двенадцать задач. Несколько правил — один раз для всех:

- **`ЗАПОЛНИТЕ`** — место для вашего кода. Пока оно в ячейке, ячейка падает с `NameError` — так и задумано.
- Последняя строка задачи кладёт ответ в `answer_N`. Её не меняйте: по ней ответы проверяются автоматически.
- Задача не получилась — замените содержимое ячейки строкой `answer_N = None`, иначе Run All остановится на ошибке. Задачи друг от друга почти не зависят: если дальше понадобится число из пропущенной задачи, его можно пересчитать своими функциями из `pilot.py`.
- При проверке ноутбук запускается ещё раз на другом варианте. Не вписывайте результаты расчётов вручную: код должен работать с данными любого варианта, а не только вашего.
- Ноутбук не скажет, верно ли вы посчитали. Итог в части 7 проверяет только формат.

В разминке код готов, не хватает одного аргумента.

### Задача 1. SRM: сошлись ли размеры групп

Прежде чем смотреть на эффект, убедимся, что с распределением клиентов всё в порядке. Мы планировали разделить пользователей 50/50. Небольшое отклонение от этой пропорции нормально, а большой перекос — повод разобраться, что произошло с экспериментом.

Посчитайте SRM двумя способами: впишите, что передать `stats.chisquare`, и сравните со своей `srm_pvalue`. Результаты должны совпасть. Семинар С8, раздел 1.

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
sizes = mine["grp"].value_counts().sort_index()
srm = stats.chisquare(sizes)

print(sizes.to_dict())
print(f"scipy:    p = {srm.pvalue:.4f}")
print(f"pilot.py: p = {srm_pvalue(sizes.tolist()):.4f}")
answer_1 = round(float(srm.pvalue), 3)
answer_1

### Задача 2. Похожи ли группы до эксперимента

Если деление случайное, до эксперимента группы тратили примерно одинаково. Проверим это по обороту за прошлые четыре недели — на него акция повлиять ещё не могла.

Сравните `spend_prev` теста и контроля t-критерием Уэлча: дисперсии групп равными не считаем. Какое значение передать в `equal_var`?

**Ответ:** p-значение, `float` с тремя знаками — например `0.123`.

In [ ]:
balance = stats.ttest_ind(test["spend_prev"], control["spend_prev"], equal_var=False)

print(f"spend_prev: test {test['spend_prev'].mean():.2f} ₽, control {control['spend_prev'].mean():.2f} ₽")
print(f"p = {balance.pvalue:.4f}")
answer_2 = round(float(balance.pvalue), 3)
answer_2

### Задача 3. Эффект и доверительный интервал

Главное число отчёта — насколько изменился средний оборот в тестовой группе относительно контрольной и в каких пределах может лежать эта разница. Посчитайте её своей `diff_ci`: разность test − control. Впишите первый аргумент. Семинар С8, раздел 2.

**Ответ:** `[разность, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
effect = diff_ci(test["spend"], control["spend"])

print(f"test {test['spend'].mean():.2f} ₽, control {control['spend'].mean():.2f} ₽")
print(f"разность {effect['diff']:.2f} ₽, интервал {effect['lo']:.2f} … {effect['hi']:.2f}, p = {effect['p']:.4f}")
answer_3 = [round(float(effect[k]), 2) for k in ("diff", "lo", "hi")]
answer_3

### Задача 4. То же в HypEx

Сверимся с HypEx: разность должна совпасть с задачей 3. Заодно посмотрим на U-критерий Манна — Уитни: он сравнивает не средние, а распределения, и на клиентов с очень большими тратами реагирует слабее.

Впишите роль колонки `grp`.

**Ответ:** `[p t-критерия, p U-критерия]`, четыре знака — например `[0.1234, 0.0123]`.

In [ ]:
ds = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole()},
    data=mine, default_role=InfoRole(),
)
ab = F(ABTest(additional_tests=["t-test", "u-test"]).execute(ds).resume)

print(ab.to_string(index=False))
answer_4 = [round(float(ab["TTest p-value"].iloc[0]), 4), round(float(ab["UTest p-value"].iloc[0]), 4)]
answer_4

**Вопрос 1.** Можно ли доверять разбиению на группы? Опирайтесь на p-значения задач 1 и 2. Как эффект из задачи 3 соотносится с MDE из дизайн-документа: исключает ли интервал ноль? Если t- и U-критерий в задаче 4 дали разные выводы, объясните, почему так могло получиться.

In [ ]:
# Числа для ответа на вопрос 1 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
q1 = pd.Series({
    "SRM, p (задача 1)": srm.pvalue,
    "баланс spend_prev, p (задача 2)": balance.pvalue,
    "эффект, ₽ (задача 3)": effect["diff"],
    "нижняя граница, ₽": effect["lo"],
    "верхняя граница, ₽": effect["hi"],
    "MDE без CUPED, ₽": mde_raw,
    "MDE с CUPED, ₽": mde_cuped,
    "эффект, % к контролю": effect["diff"] / control["spend"].mean() * 100,
    "p t-критерия (задача 4)": ab["TTest p-value"].iloc[0],
    "p U-критерия (задача 4)": ab["UTest p-value"].iloc[0],
})
print(q1.round(4).to_string())
print("интервал исключает ноль:", effect["lo"] > 0, "· эффект больше MDE без CUPED:", effect["diff"] > mde_raw)

# Почему t и U могут разойтись: где по распределению сидит разница
qs = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
quant = pd.DataFrame({"control": control["spend"].quantile(qs), "test": test["spend"].quantile(qs)})
quant["test − control"] = quant["test"] - quant["control"]
print(quant.round(0).to_string())

#### ✍️ Ответ на вопрос 1

Разбиению доверять можно: SRM p = 0,454 (2 971 в тесте против 3 029 в контроле), а до эксперимента оборот групп почти одинаковый — 3 228 и 3 169 ₽, p = 0,479. Эффект +199 ₽ (+6,2 %), интервал от 29 до 369 ₽ — ноль он исключает (p = 0,022), но сама оценка меньше MDE без CUPED (232 ₽), а нижняя граница почти у нуля, так что эффект может быть и совсем маленьким. t-критерий значим (p = 0,022), а U-критерий нет (p = 0,322): медиана оборота в тесте почти не сдвинулась (+13 ₽), а разница сидит в верхнем хвосте — +697 ₽ на 95-м перцентиле и +2 281 ₽ на 99-м. То есть среднее тянут вверх несколько крупных клиентов, а у типичного подписчика заметного сдвига нет — U-критерий на таких клиентов реагирует слабо.

## Часть 4. Дописываем код

Теперь не хватает целой строки.

### Задача 5. Метрики-стражи: отток и обращения

Рост оборота мало что значит, если акция заодно уводит клиентов из подписки или нагружает поддержку. Отток и число обращений — наши метрики-стражи. Для них подходит тот же `diff_ci`, а для оттока, раз это «да/нет», можно посчитать ещё и χ² по таблице частот, как в разделе 3 семинара С8.

Допишите строку: p-значение χ² по таблице `table` — в переменную `p_churn`.

**Ответ:** `[разница оттока в п.п., p χ², разница обращений на клиента, p обращений]` — два, три, три и три знака, например `[0.86, 0.857, 0.027, 0.743]`.

In [ ]:
guards = pd.DataFrame([
    {"metric": metric, "control": control[metric].mean(), "test": test[metric].mean(),
     **diff_ci(test[metric], control[metric])}
    for metric in ("churned", "tickets")
]).set_index("metric")
print(guards.round(4).to_string())

table = pd.crosstab(mine["grp"], mine["churned"])
print(table)
p_churn = stats.chi2_contingency(table).pvalue

print(f"χ² по оттоку: p = {p_churn:.4f}")
answer_5 = [round(float(guards.loc["churned", "diff"] * 100), 2), round(float(p_churn), 3),
            round(float(guards.loc["tickets", "diff"]), 3), round(float(guards.loc["tickets", "p"]), 3)]
answer_5

### Задача 6. Поправка на множественные сравнения

Мы смотрим сразу на четыре метрики: оборот, покупки, отток и обращения. Чем больше сравнений, тем выше шанс, что какое-то окажется «значимым» случайно. На семинаре С8 поправку Холма считал HypEx — но там было три группы, а у нас две. Посмотрите, что HypEx напечатает про поправку теперь.

Допишите строку: p-значения с поправкой Холма — в колонку `holm`. Считаем через `statsmodels`: `multipletests(..., method="holm")` возвращает кортеж, нужный массив — второй элемент.

**Ответ:** список метрик, значимых после поправки, — например `["churned", "tickets"]`; если таких нет, `[]`.

In [ ]:
from statsmodels.stats.multitest import multipletests

roles_all = {"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
             "purchases": TargetRole(), "churned": TargetRole(), "tickets": TargetRole()}
res_all = ABTest(multitest_method=ABNTestMethodsEnum.holm).execute(
    Dataset(roles=roles_all, data=mine, default_role=InfoRole()))
print("HypEx о поправке:", res_all.multitest)

metrics = F(res_all.resume)[["feature", "control mean", "test mean", "difference", "TTest p-value"]].copy()
metrics["holm"] = multipletests(metrics["TTest p-value"], method="holm")[1]
metrics["rejected"] = metrics["holm"] < 0.05

print(metrics.round(4).to_string(index=False))
answer_6 = sorted(metrics.loc[metrics["rejected"], "feature"].tolist())
answer_6

### Задача 7. CUPED

В задаче 3 мы посчитали эффект обычным способом. Но клиенты сильно различаются по обороту: кто-то тратит пару сотен рублей, кто-то — десятки тысяч. Попробуем CUPED и посмотрим, получится ли уменьшить дисперсию и получить более узкий доверительный интервал.

В качестве ковариаты возьмите оборот за предыдущие четыре недели, `spend_prev`. Допишите строку: примените свою `cuped(y, x)` — результат в `mine["spend_cuped"]` и `theta`. Семинар С8, разделы 5 и 8.

**Ответ:** `[θ, снижение дисперсии в %, разность, нижняя граница, верхняя граница]` — три, один и по два знака, например `[0.543, 21.0, 123.45, 12.34, 234.56]`.

In [ ]:
# Ваша cuped() из pilot.py: метрика spend, ковариата spend_prev -> mine["spend_cuped"] и theta.
mine["spend_cuped"], theta = cuped(mine["spend"], mine["spend_prev"])
reduction = 1 - mine["spend_cuped"].var() / mine["spend"].var()

g = mine.groupby("grp")
effect_cuped = diff_ci(g.get_group("test")["spend_cuped"], g.get_group("control")["spend_cuped"])

print(f"theta = {theta:.3f}, дисперсия ниже на {reduction:.1%}")
print(f"разность {effect_cuped['diff']:.2f} ₽, интервал {effect_cuped['lo']:.2f} … {effect_cuped['hi']:.2f}")
answer_7 = [round(float(theta), 3), round(float(reduction * 100), 1),
            *[round(float(effect_cuped[k]), 2) for k in ("diff", "lo", "hi")]]
answer_7

### Задача 8. CUPED в HypEx

Тот же расчёт средствами HypEx. Оценка должна совпасть с задачей 7; если не совпала, ошибка в одном из двух мест — лучше найти её до отчёта.

Допишите строку: роль `pre_role` для `spend_prev` — значение метрики `spend` на один период раньше. Семинар С8, раздел 5.

**Ответ:** разность после CUPED, рубли с двумя знаками — например `123.45`.

In [ ]:
# Роль для spend_prev: предпериод метрики spend, на один период назад.
pre_role = PreTargetRole(parent="spend", lag=1)

ds_cuped = Dataset(
    roles={"client_id": InfoRole(str), "grp": TreatmentRole(), "spend": TargetRole(),
           "spend_prev": pre_role},
    data=mine[["client_id", "grp", "spend", "spend_prev"]], default_role=InfoRole(),
)
res_cuped = ABTest(cuped_features={"spend": "spend_prev"}).execute(ds_cuped)

print(F(res_cuped.cuped.resume).to_string(index=False))
cuped_table = F(res_cuped.resume)
print(cuped_table.to_string(index=False))
answer_8 = round(float(cuped_table.loc[cuped_table["feature"] == "spend_cuped", "difference"].iloc[0]), 2)
answer_8

**Вопрос 2.** Что изменилось после проверки стражей и CUPED? Сработал ли какой-то страж (задача 5) и что осталось значимым после поправки (задача 6)? На сколько процентов сузился интервал после CUPED (задачи 3 и 7)? Изменилась ли сама оценка эффекта? Сравните средний оборот прошлого периода в тесте и контроле и попробуйте объяснить результат.

In [ ]:
# Числа для ответа на вопрос 2 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
width_raw = effect["hi"] - effect["lo"]
width_cuped = effect_cuped["hi"] - effect_cuped["lo"]
prev_gap = test["spend_prev"].mean() - control["spend_prev"].mean()
q2 = pd.Series({
    "отток, разница п.п.": guards.loc["churned", "diff"] * 100,
    "отток, интервал от, п.п.": guards.loc["churned", "lo"] * 100,
    "отток, интервал до, п.п.": guards.loc["churned", "hi"] * 100,
    "обращения, разница на клиента": guards.loc["tickets", "diff"],
    "значимых после Холма": len(answer_6),
    "мин. p после Холма": metrics["holm"].min(),
    "ширина интервала без CUPED, ₽": width_raw,
    "ширина интервала с CUPED, ₽": width_cuped,
    "интервал уже на, %": (1 - width_cuped / width_raw) * 100,
    "оценка без CUPED, ₽": effect["diff"],
    "оценка с CUPED, ₽": effect_cuped["diff"],
    "spend_prev: test − control, ₽": prev_gap,
    "θ × разница spend_prev, ₽": theta * prev_gap,
})
print(q2.round(4).to_string())

#### ✍️ Ответ на вопрос 2

Ни один страж формально не сработал: отток в тесте выше на 0,41 п.п. (интервал от −0,56 до +1,37 п.п., p χ² = 0,448), обращений чуть меньше — −0,003 на клиента (p = 0,636), но вред по оттоку до +1,37 п.п. интервал не исключает. После поправки Холма на четыре метрики значимым не осталось ничего: лучший p — 0,086 у оборота, поэтому ответ задачи 6 пустой. CUPED сузил интервал на 20 % (с 340 до 271 ₽), а оценку снизил с 199 до 162 ₽: тестовая группа ещё до эксперимента тратила на 59 ₽ больше контроля, и θ = 0,628 × 59 ≈ 37 ₽ — ровно эта разница. Значит, часть «эффекта» из задачи 3 — случайный перекос групп, а не новая главная.

## Часть 5. Самостоятельные задачи

Код пишете целиком — от `mine`, `test` и `control` до ответа. Ответ кладите обычным `float` или `str`, а не `numpy`-числом: `round(float(...), 2)`.

### Задача 9. Бутстреп

До этого мы строили доверительный интервал через нормальное приближение. Но оборот распределён неравномерно: много клиентов с небольшими тратами и немного — с очень большими. Попробуем другой способ — бутстреп: будем многократно составлять новые выборки из имеющихся наблюдений и каждый раз пересчитывать разность средних.

1. `rng = np.random.default_rng(VARIANT)` — номер варианта как seed, чтобы повторный запуск дал те же числа.
2. 2 000 раз: выберите с возвращением из оборотов теста столько значений, сколько клиентов в тесте (`rng.choice(x, size=len(x))`), затем так же из контроля, и запишите разность средних. Сначала тест, потом контроль — тогда результат воспроизводится.
3. Интервал — 2,5-й и 97,5-й перцентили этих разностей (`np.percentile`).

Сравните получившийся интервал с задачей 3. Насколько они отличаются?

**Ответ:** `[нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[12.34, 234.56]`.

In [ ]:
# Задача 9. 95 % интервал разности средних оборота test − control бутстрепом:
# 2 000 повторов, rng = np.random.default_rng(VARIANT), сначала тест, потом контроль.
rng = np.random.default_rng(VARIANT)
x_t = test["spend"].to_numpy()
x_c = control["spend"].to_numpy()

boot = np.empty(2000)
for i in range(2000):
    s_t = rng.choice(x_t, size=len(x_t))     # сначала тест
    s_c = rng.choice(x_c, size=len(x_c))     # потом контроль
    boot[i] = s_t.mean() - s_c.mean()

boot_lo, boot_hi = np.percentile(boot, [2.5, 97.5])
print(f"бутстреп: {boot_lo:.2f} … {boot_hi:.2f} ₽, ширина {boot_hi - boot_lo:.2f}")
print(f"задача 3: {effect['lo']:.2f} … {effect['hi']:.2f} ₽, ширина {effect['hi'] - effect['lo']:.2f}")
answer_9 = [round(float(boot_lo), 2), round(float(boot_hi), 2)]
answer_9

### Задача 10. Какую ковариату взять

Для CUPED годится любой признак, измеренный до начала эксперимента. Чем сильнее он связан с метрикой, тем сильнее падает дисперсия: доля снижения равна ρ², квадрату корреляции. В задаче 7 мы взяли оборот за прошлые четыре недели — просто потому, что так делали на семинаре. Проверим, нет ли варианта лучше.

Посчитайте ρ² с `spend` в процентах для шести кандидатов: `"spend_prev"`, `"spend_prev2"`, `"spend_prev+spend_prev2"` (сумма двух периодов), `"purchases_prev"`, `"sessions_prev"`, `"sessions_prev2"`. Отдельно — для `purchases`, покупок за время эксперимента. Ответ — лучший из шести. Почему `purchases` брать нельзя, объясните в вопросе 3: на семинаре С8 эта ловушка уже встречалась.

**Ответ:** `[название ковариаты, ρ² в %]`, название — как в списке, процент с одним знаком, например `["purchases_prev2", 12.3]`.

In [ ]:
# Задача 10. ρ² с spend (в %) для шести кандидатов из предпериода и отдельно для purchases.
# Ответ — лучший кандидат из предпериода и его ρ² в процентах.
candidates = {
    "spend_prev": mine["spend_prev"],
    "spend_prev2": mine["spend_prev2"],
    "spend_prev+spend_prev2": mine["spend_prev"] + mine["spend_prev2"],
    "purchases_prev": mine["purchases_prev"],
    "sessions_prev": mine["sessions_prev"],
    "sessions_prev2": mine["sessions_prev2"],
}
rho2 = pd.Series({name: mine["spend"].corr(x) ** 2 * 100 for name, x in candidates.items()})
rho2_purchases = mine["spend"].corr(mine["purchases"]) ** 2 * 100

print(rho2.sort_values(ascending=False).round(1).to_string())
print(f"purchases (за время эксперимента, брать нельзя): {rho2_purchases:.1f}")

best_cov = rho2.idxmax()
answer_10 = [best_cov, round(float(rho2[best_cov]), 1)]
answer_10

### Задача 11. Мэтчинг на тех же данных

У нас рандомизированный эксперимент, поэтому для оценки эффекта мэтчинг, строго говоря, не нужен. Но интересно посмотреть, какой результат он даст на тех же данных: подберём каждому клиенту похожих из другой группы и сравним оценку с обычной.

Используйте `Matching(n_neighbors=5)` и признаки, измеренные до начала эксперимента: `age`, `tenure_days`, `kyc_level`, `members`, `spend_prev`, `sessions_prev`. Группу HypEx ждёт числом — заведите колонку `t`: 1 у теста, 0 у контроля. Семинар С8, раздел 6.

**Ответ:** `[ATE, нижняя граница, верхняя граница]`, рубли с двумя знаками — например `[123.45, 12.34, 234.56]`.

In [ ]:
# Задача 11. Эффект на spend мэтчингом: Matching(n_neighbors=5),
# признаки age, tenure_days, kyc_level, members, spend_prev, sessions_prev.
# Ответ — ATE, нижняя и верхняя граница интервала.
feats = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]
mine_t = mine.assign(t=(mine["grp"] == "test").astype(int))

ds_match = Dataset(
    roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
           **{f: FeatureRole() for f in feats}},
    data=mine_t[["client_id", "t", "spend", *feats]], default_role=InfoRole(),
)
res_match = Matching(n_neighbors=5).execute(ds_match)
match = F(res_match.resume)
print(match.to_string())

ate = match.loc["ATE"]
answer_11 = [round(float(ate["Effect Size"]), 2), round(float(ate["CI Lower"]), 2), round(float(ate["CI Upper"]), 2)]
answer_11

### Задача 12. Что это в деньгах

Продакта интересует не p-значение, а деньги. Точных ставок в базе нет, поэтому берём допущения — они уже стоят в ячейке, их не меняйте.

Посчитайте результат акции на одного клиента за четыре недели: комиссия с прироста оборота (эффект с CUPED из задачи 7 × `COMMISSION`) минус дополнительные обращения (разница из задачи 5 × `TICKET_COST`) минус дополнительный отток (разница из задачи 5 × `CHURN_COST`). Если обращений или оттока в тесте меньше, знак поменяется сам. Нужные числа можно взять из задач 5 и 7 или пересчитать здесь же своими `cuped` и `diff_ci`. Напечатайте все слагаемые и итог.

**Ответ:** рубли на клиента с двумя знаками — например `-1.23`.

In [ ]:
# Задача 12. Результат акции в рублях на клиента за 4 недели.
# Допущения — не меняйте их.
COMMISSION = 0.015    # комиссия ПРАЙМ с оборота в партнёрских сервисах
TICKET_COST = 150     # стоимость одного обращения в поддержку, ₽
CHURN_COST = 750      # потеря от ушедшего подписчика: три месяца «Базового», ₽

# Эффект с CUPED (как в задаче 7) и разницы стражей (как в задаче 5) — пересчитываем здесь.
spend_adj, _ = cuped(mine["spend"], mine["spend_prev"])
is_test = mine["grp"] == "test"
lift = diff_ci(spend_adj[is_test], spend_adj[~is_test])["diff"]
ticket_gap = diff_ci(test["tickets"], control["tickets"])["diff"]
churn_gap = diff_ci(test["churned"], control["churned"])["diff"]

money = pd.Series({
    "комиссия с прироста оборота": lift * COMMISSION,
    "дополнительные обращения": -ticket_gap * TICKET_COST,
    "дополнительный отток": -churn_gap * CHURN_COST,
})
money["итого на клиента за 4 недели"] = money.sum()
print(f"прирост оборота с CUPED {lift:.2f} ₽, обращений {ticket_gap:+.4f}, оттока {churn_gap * 100:+.2f} п.п.")
print(money.round(2).to_string())
answer_12 = round(float(money["итого на клиента за 4 недели"]), 2)
answer_12

**Вопрос 3.** Какой оценке эффекта вы доверяете больше всего? Сравните ширину интервалов из задач 3, 7, 9 и 11 и объясните разницу. Почему `purchases` из задачи 10 нельзя брать ковариатой, хотя ρ² у неё самый большой? Окупается ли акция при допущениях задачи 12 и чего в этом расчёте не хватает?

In [ ]:
# Числа для ответа на вопрос 3 считайте здесь — от загруженных данных.
# Ячейка тоже выполняется на контрольном варианте: чисел руками не вписывайте.
intervals = pd.DataFrame({
    "задача 3: разность средних": [effect["diff"], effect["lo"], effect["hi"]],
    "задача 7: CUPED": [effect_cuped["diff"], effect_cuped["lo"], effect_cuped["hi"]],
    "задача 9: бутстреп": [boot.mean(), boot_lo, boot_hi],
    "задача 11: мэтчинг": [ate["Effect Size"], ate["CI Lower"], ate["CI Upper"]],
}, index=["оценка", "от", "до"]).T
intervals["ширина"] = intervals["до"] - intervals["от"]
print(intervals.round(2).to_string())

# CUPED по лучшей ковариате из задачи 10 — насколько ещё сузился бы интервал
best_adj, best_theta = cuped(mine["spend"], candidates[best_cov])
best_eff = diff_ci(best_adj[mine["grp"] == "test"], best_adj[mine["grp"] == "control"])
print(f"CUPED по {best_cov}: {best_eff['diff']:.2f} ₽, {best_eff['lo']:.2f} … {best_eff['hi']:.2f}, "
      f"ширина {best_eff['hi'] - best_eff['lo']:.2f}")
print(f"ρ² purchases {rho2_purchases:.1f} % против лучшего из предпериода {rho2.max():.1f} %")
print(f"деньги на клиента за 4 недели: {money.round(2).to_dict()}")
print(f"на {len(mine)} клиентов за 4 недели: {money.iloc[-1] * len(mine):.0f} ₽")

churn_lo, churn_hi = diff_ci(test["churned"], control["churned"])["lo"], diff_ci(test["churned"], control["churned"])["hi"]
print(f"отток в деньгах по краям интервала: от {-churn_hi * CHURN_COST:.2f} до {-churn_lo * CHURN_COST:.2f} ₽ на клиента")

#### ✍️ Ответ на вопрос 3

Больше всего доверяю CUPED: +162 ₽, интервал 27–298 ₽ шириной 271 ₽ — он самый узкий и учитывает перекос в предпериоде, а ковариата выбрана заранее; бутстреп (31–368, ширина 337) почти совпал с обычным интервалом (29–369, ширина 340) — при 3 000 клиентов нормальное приближение держится, а мэтчинг (233 ₽, 91–375, ширина 284) при случайном делении добавляет свой шум подбора и сдвинул оценку вверх. `purchases` брать нельзя, хотя у неё ρ² = 70,2 % против 43,7 % у лучшей ковариаты `sessions_prev`: покупки измерены во время эксперимента и сами выросли от акции, поэтому CUPED вычел бы часть эффекта вместе с шумом. При допущениях задачи 12 акция не окупается: +2,43 ₽ комиссии и +0,44 ₽ на обращениях съедает −3,05 ₽ от оттока, итого −0,17 ₽ на клиента (около −1 000 ₽ на 6 000 клиентов за 4 недели), причём из-за широкого интервала оттока его слагаемое может быть от −10,28 до +4,18 ₽. В расчёте не хватает стоимости разработки главной, того, сколько эффект продержится после 4 недель, неопределённости остальных слагаемых, а ещё того, что оценка зависит от ковариаты: CUPED по `sessions_prev` дал бы 245 ₽ вместо 162.

## Часть 6. Отчёт: что скажем продакту?

Представьте, что продакт написал вам в чат: «Ну что там с экспериментом? Запускаем на всех?»

Результаты всех проверок у вас есть, теперь их нужно собрать в одно решение. Напишите короткий отчёт на четыре абзаца. Не пересказывайте каждую задачу и не объясняйте, что такое CUPED: продакту важно понять, какой эффект получился, насколько мы в нём уверены, какие есть риски и что делать дальше. Числа — из ноутбука, эффект — с интервалом.

Можно рекомендовать запуск, доработку с повторным экспериментом или отказ от акции. Главное — чтобы рекомендация следовала из ваших расчётов и решающего правила из дизайн-документа. Образец — раздел 7 семинара С8.

#### ✍️ Отчёт продакту

**Что проверяли:** новая главная страница приложения 4 недели была у половины из 6 000 подписчиков (2 971 в тесте, 3 029 в контроле). Ждали, что оборот в партнёрских сервисах вырастет примерно на 190 ₽ на клиента, а отток и обращения в поддержку не вырастут; запускать договорились, только если рост значим с поправкой на четыре метрики, стражи в норме и в деньгах выходит не меньше нуля. Деление прошло честно: размеры групп и траты до эксперимента не различаются (p = 0,45 и 0,48).

**Что получилось:** оборот в тесте выше на 162 ₽ на клиента за 4 недели (+5 %), с учётом трат до эксперимента; интервал от 27 до 298 ₽. Без поправки на четыре метрики это значимо, с поправкой — нет (p = 0,086). Рост дают в основном самые крупные клиенты: у типичного подписчика оборот почти не изменился (медиана +13 ₽). Отток в тесте выше на 0,4 п.п. (3,97 % против 3,57 %) — это может быть случайностью, но рост до +1,4 п.п. данные не исключают; обращений не стало больше. В деньгах комиссия с прироста оборота (+2,4 ₽ на клиента) примерно уходит на отток (−3,1 ₽): итог −0,17 ₽ на клиента, то есть около нуля.

**Какие остались вопросы:** не понятно, вредит ли новая главная удержанию: 6 000 клиентов слишком мало, чтобы заметить рост оттока на полпроцента. Неясно, продержится ли рост оборота дольше 4 недель или это эффект новизны. Расчёт денег держится на допущениях (комиссия 1,5 %, ушедший подписчик — 750 ₽, обращение — 150 ₽) и не учитывает стоимость разработки. Эксперимент изначально мог заметить только эффект от 188 ₽, а рост на 3 % (96 ₽) — нет: для него нужно около 11 тыс. клиентов на группу.

**Что рекомендуем:** на всех пока не запускаем — по нашему же правилу это «дорабатываем»: рост оборота не прошёл поправку, а отток не исключает вреда. Предлагаю посмотреть, почему ушли клиенты из теста (что их могло запутать на новой главной), и повторить тест на большей выборке — от 11 тыс. подписок на группу, с CUPED и заранее выбранной ковариатой, — отдельно следя за оттоком. Если рост подтвердится, а отток останется на уровне контроля, раскатываем.

## Часть 7. Итог

Готовая ячейка: печатает ваши ответы и проверяет их формат, но не правильность. Последняя строка служебная — по ней ответы сверяются автоматически. Посмотрите на вывод перед сдачей.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 13)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте текст: дизайн-документ, ответы на вопросы 1–3 и отчёт продакту.")
print("\n--- служебная строка, не удаляйте ---")
print("HW3_ANSWERS=" + json.dumps(
    {"variant": VARIANT, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 8. Исследование — по желанию

Это возможность попробовать небольшое настоящее исследование, а не решить ещё несколько задач: готового формата ответа и заготовок нет, что считать — решаете вы. Каждая часть даёт до 1 балла сверх восьми и засчитывается, если основная часть набрала не меньше 15 баллов из 20.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется целиком, в том числе на контрольном варианте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Всё считайте от `mine`, случайность заводите через `np.random.default_rng(VARIANT)`. Текст вывода, как и везде, пишете сами.

### Исследование А. Без рандомизации

Представьте, что эксперимента не было: акцию просто показали в приложении, и увидели её только активные клиенты. Продакт сравнил их со всеми остальными и получил огромный эффект. В жизни узнать, насколько он завышен, нельзя. А у вас есть рандомизированные данные и правильный ответ можно посчитать — значит, можно проверить, насколько наблюдательные данные искажают оценку и помогает ли мэтчинг.

1. Соберите «наблюдательные» данные: из теста — только клиентов с `sessions_prev` не ниже медианы по вашему варианту, из контроля — всех.
2. Посчитайте наивную разность средних `spend`.
3. Оцените эффект мэтчингом по признакам задачи 11. Здесь нужен ATT — эффект на тех, кто акцию увидел.
4. Посчитайте эталон: разность между активными из теста и активными из контроля. Эту оценку даёт рандомизация.
5. Уберите `sessions_prev` из признаков мэтчинга и повторите. Что изменилось и почему?
6. Когда мэтчингу можно доверять — и чего ваш расчёт не доказывает?

Семинар С8, раздел 6; [слайды 22–27 Л8](../../08-анализ-пилота/лекция/Л8-слайды.pdf).

In [ ]:
# Исследование А: ваш код. Всё — от mine.
feats_a = ["age", "tenure_days", "kyc_level", "members", "spend_prev", "sessions_prev"]
med_sessions = mine["sessions_prev"].median()
active = mine["sessions_prev"] >= med_sessions

# 1. «Наблюдательные» данные: из теста — только активные, из контроля — все
obs = mine[(mine["grp"] == "control") | ((mine["grp"] == "test") & active)].copy()
obs["t"] = (obs["grp"] == "test").astype(int)

# 2. Наивная разность
naive_a = obs.loc[obs["t"] == 1, "spend"].mean() - obs.loc[obs["t"] == 0, "spend"].mean()


def att_matching(data: pd.DataFrame, features: list[str]) -> pd.Series:
    """ATT мэтчингом HypEx (5 соседей) по заданным признакам."""
    ds_a = Dataset(
        roles={"client_id": InfoRole(str), "t": TreatmentRole(), "spend": TargetRole(),
               **{f: FeatureRole() for f in features}},
        data=data[["client_id", "t", "spend", *features]], default_role=InfoRole(),
    )
    return F(Matching(n_neighbors=5).execute(ds_a).resume).loc["ATT"]


# 3. Мэтчинг по признакам задачи 11
att_full = att_matching(obs, feats_a)

# 4. Эталон: активные из теста против активных из контроля — это даёт рандомизация
truth_a = diff_ci(mine.loc[active & (mine["grp"] == "test"), "spend"],
                  mine.loc[active & (mine["grp"] == "control"), "spend"])

# 5. Без sessions_prev — признака, по которому шёл отбор
att_nosess = att_matching(obs, [f for f in feats_a if f != "sessions_prev"])

# Насколько отобранные отличались от контроля до эксперимента
prev_profile = (obs.groupby("t")[["sessions_prev", "spend_prev", "age"]].mean()
                   .rename(index={0: "control, все", 1: "test, активные"}))
corr_sess = mine["sessions_prev"].corr(mine["spend_prev"])

research_a = pd.DataFrame({
    "оценка": [naive_a, att_full["Effect Size"], att_nosess["Effect Size"], truth_a["diff"]],
    "от": [np.nan, att_full["CI Lower"], att_nosess["CI Lower"], truth_a["lo"]],
    "до": [np.nan, att_full["CI Upper"], att_nosess["CI Upper"], truth_a["hi"]],
}, index=["наивная разность", "мэтчинг ATT, все 6 признаков", "мэтчинг ATT, без sessions_prev",
          "эталон: активные test − активные control"])
research_a["смещение к эталону"] = research_a["оценка"] - truth_a["diff"]
print(f"медиана sessions_prev: {med_sessions:.0f}; активных в тесте: {int((obs['t'] == 1).sum())}, "
      f"контроль: {int((obs['t'] == 0).sum())}; корреляция sessions_prev и spend_prev: {corr_sess:.2f}")
print(prev_profile.round(1).to_string())
print(research_a.round(2).to_string())

#### ✍️ Исследование А

**Вопрос.** Если бы акцию увидели только активные клиенты и эксперимента не было, насколько наивное сравнение завысило бы эффект и спасает ли мэтчинг?

**Как проверял.** Активные — клиенты с `sessions_prev` не ниже медианы (15 сессий). «Наблюдательные» данные: 1 498 активных из теста против всех 3 029 из контроля. Посчитал наивную разность средних `spend`, ATT мэтчингом HypEx (5 соседей) по шести признакам задачи 11 и без `sessions_prev`. Эталон — разность между активными из теста и активными из контроля: её даёт рандомизация.

**Что получилось.** Отобранные клиенты и до акции сильно отличались от контроля: 28,1 сессии против 18,7, оборот 4 726 ₽ против 3 169 ₽. Наивная разность — 1 715 ₽ при эталоне 355 ₽ (интервал 74–636): завышение почти в 5 раз, на 1 360 ₽. Мэтчинг по всем признакам дал 387 ₽ (163–611) — смещение всего +32 ₽, внутри интервала эталона. Без `sessions_prev` мэтчинг дал 736 ₽ (504–968): смещение +381 ₽, и интервал эталон уже не накрывает. Совсем до наивной разности оценка не выросла, потому что `spend_prev` связан с сессиями (корреляция 0,70) и часть отбора ловит сам.

**Что это значит.** Мэтчингу можно доверять, только когда все признаки, по которым шёл отбор, есть в данных и попали в подбор: здесь я знал правило отбора и положил его в признаки — и оценка сошлась. Пропусти один такой признак — и оценка смещена, а интервал при этом узкий и выглядит уверенно. Чего расчёт не доказывает: это один вариант и одно известное правило отбора; в жизни правило неизвестно, и проверить, что ненаблюдаемых причин нет, по данным нельзя. К тому же сам эталон шумный (интервал шириной 560 ₽), так что «совпало» значит «не отличается в пределах шума», а не «точно равно».

### Исследование Б. Сколько клиентов было нужно

В дизайн-документе вы посчитали MDE по формуле. Формула опирается на нормальное приближение, а у оборота тяжёлый хвост. Проверьте её симуляцией, как в разделе 5 семинара С7, и ответьте: сколько клиентов нужно, чтобы заметить рост оборота на 3 %?

1. Возьмите оборот контрольной группы — эффекта там нет.
2. Для нескольких размеров группы n (например, 1 000, 3 000, 10 000, 20 000, 40 000) 500 раз: выберите с возвращением 2n значений, к первой половине прибавьте эффект — 3 % от среднего оборота контроля — и сравните половины t-критерием Уэлча. Доля отвержений при α = 0,05 — это мощность.
3. Повторите для оборота после CUPED. Сначала посчитайте скорректированный оборот для каждого клиента контроля (θ по контрольной группе, ковариата — `spend_prev`), а потом выбирайте с возвращением уже его. Если выбирать `spend` и `spend_prev` по отдельности, связь между ними пропадёт, и CUPED ничего не даст.
4. Сравните с формулой мощности из семинара С7 (★3).
5. С какого n мощность доходит до 80 % без CUPED и с ним? Хватило бы вашего эксперимента? Где симуляция расходится с формулой и почему?

In [ ]:
# Исследование Б: ваш код. Всё — от mine.
rng_b = np.random.default_rng(VARIANT)
alpha_b, reps = 0.05, 500

y_c = control["spend"]
effect_b = 0.03 * y_c.mean()                                     # +3 % к среднему обороту контроля
y_c_cuped, theta_c = cuped(y_c, control["spend_prev"])           # θ — по контрольной группе

samples = {"без CUPED": y_c.to_numpy(), "с CUPED": y_c_cuped.to_numpy()}
sigmas = {name: float(np.std(arr, ddof=1)) for name, arr in samples.items()}
n_formula = {name: n_per_group(s, effect_b) for name, s in sigmas.items()}


def sim_power(arr: np.ndarray, n: int, eff: float) -> float:
    """Доля отвержений H0 (α = 0,05) Уэлчем, когда к первой половине прибавлен eff."""
    hits = 0
    for _ in range(reps):
        draw = rng_b.choice(arr, size=2 * n)
        hits += stats.ttest_ind(draw[:n] + eff, draw[n:], equal_var=False).pvalue < alpha_b
    return hits / reps


def formula_power(sigma: float, n: int) -> float:
    """Мощность по формуле С7 ★3: Φ(MDE / (σ√(2/n)) − z_{1−α/2})."""
    return float(stats.norm.cdf(effect_b / (sigma * (2 / n) ** 0.5) - stats.norm.ppf(1 - alpha_b / 2)))


grid = sorted({1_000, n_each, 5_000, 10_000, 15_000, 20_000, 30_000, 40_000})
rows = []
for n in grid:
    row = {"n на группу": n}
    for name, arr in samples.items():
        row[f"симуляция, {name}"] = sim_power(arr, n, effect_b)
        row[f"формула, {name}"] = formula_power(sigmas[name], n)
    rows.append(row)
power_table = pd.DataFrame(rows).set_index("n на группу")

# Ошибка I рода при нашем размере: тот же генератор, эффект 0
alpha_sim = {name: sim_power(arr, n_each, 0.0) for name, arr in samples.items()}

print(f"эффект +3 %: {effect_b:.0f} ₽; σ без CUPED {sigmas['без CUPED']:.0f} ₽, с CUPED {sigmas['с CUPED']:.0f} ₽, "
      f"θ = {theta_c:.3f}")
print(f"нужно на группу по формуле n_per_group: без CUPED {n_formula['без CUPED']}, с CUPED {n_formula['с CUPED']}")
print(f"ошибка I рода при n = {n_each}: без CUPED {alpha_sim['без CUPED']:.1%}, с CUPED {alpha_sim['с CUPED']:.1%}")
print(f"асимметрия оборота контроля: {stats.skew(samples['без CUPED']):.1f}, после CUPED: {stats.skew(samples['с CUPED']):.1f}")
print(power_table.round(3).to_string())

#### ✍️ Исследование Б

**Вопрос.** Сколько клиентов на группу нужно, чтобы с мощностью 80 % заметить рост оборота на 3 %, и не обманывает ли формула мощности на обороте с тяжёлым хвостом?

**Как проверял.** Взял оборот контрольной группы (эффекта там нет); +3 % — это 96 ₽. Для n от 1 000 до 40 000 на группу 500 раз выбирал с возвращением 2n клиентов, к первой половине прибавлял 96 ₽ и сравнивал половины Уэлчем при α = 0,05; доля отвержений — мощность. То же для оборота после CUPED (θ = 0,591 по контролю, ковариата `spend_prev`, корректировал каждого клиента до выборки). Сравнил с формулой ★3 из С7 и с `n_per_group`. Отдельно проверил ошибку I рода при нашем n = 3 000 (эффект 0).

**Что получилось.** По формуле нужно 16 477 клиентов на группу без CUPED и 10 844 с CUPED (σ падает с 3 108 до 2 521 ₽). Симуляция согласна: без CUPED 77,8 % при 15 000 и 87,8 % при 20 000; с CUPED 78,2 % при 10 000 и 90,6 % при 15 000. Расхождение с формулой везде не больше 2–3 п.п., а это и есть шум 500 повторов (±1,8 п.п.). При наших 3 000 на группу мощность всего 23 % без CUPED и 32 % с CUPED, а ошибка I рода честная — 5,0 % и 4,6 %.

**Что это значит.** Нашего эксперимента не хватило бы: рост на 3 % он увидел бы примерно в одном случае из трёх-четырёх, нужно было в 3,5 раза больше клиентов даже с CUPED. Тяжёлый хвост оборота (асимметрия 2,3) при n от 1 000 формулу не ломает: среднее по тысячам клиентов уже распределено почти нормально. Чего расчёт не доказывает: я прибавлял всем одинаковые 96 ₽, а в вопросе 1 видно, что рост сидит в хвосте — при таком эффекте разброс в тесте больше и мощность может быть ниже; кроме того, выборка — только контроль одного варианта, а θ оценён на тех же данных, по которым шла симуляция.